# Notebook 01 - BS-80K: training the four backbones (metastasis vs normal)
BS-80K is a public whole-body bone-scan dataset (3,247 patients, normal vs metastasis). Only the whole-body anterior and posterior images are used. The trained backbones serve as the BS-80K initialisation in all later notebooks.
Patient-wise 80/10/10 split; augmentation on the training split only.

## Setup (GPU, libraries, Drive)

In [ ]:
!pip install -q timm kagglehub
import os, sys, re, glob
import torch
from google.colab import drive
drive.mount('/content/drive')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND - select Runtime > Change runtime type > T4 GPU')

## Settings
Run first with `QUICK_TEST = True` (small subset, one model, one epoch) to check the pipeline, then set `QUICK_TEST = False` and run all. Each model is saved as soon as it is finished; re-running skips finished models.

In [ ]:
QUICK_TEST = True

BASE = '/content/drive/MyDrive/Colab Notebooks/bonescan'
OUT = BASE + '/bs80k'
os.makedirs(OUT, exist_ok=True)

MODELS = ['tf_efficientnetv2_s', 'resnext50_32x4d', 'regnety_016',
          'mobilenetv3_large_100', 'inception_v3']
EPOCHS = 20        # BS-80K is large; 20 epochs are sufficient
PATIENCE = 5
BATCH = 16
SEED = 42
H, W = 512, 176    # apne data wala hi size

if QUICK_TEST:
    MODELS = ['mobilenetv3_large_100']
    EPOCHS = 1
print('Models:', MODELS, '| Epochs:', EPOCHS, '| Quick test:', QUICK_TEST)

## Download BS-80K and locate the whole-body folders
The dataset is downloaded from Kaggle (several GB).

In [ ]:
import kagglehub
DATA = kagglehub.dataset_download('mariusmarin/bs-80k')
print('Downloaded to:', DATA)

def find_dir(name):
    for root, dirs, _ in os.walk(DATA):
        for d in dirs:
            if d.lower() == name.lower():
                return os.path.join(root, d)
    return None

ANT_DIR, POST_DIR = find_dir('wholeBodyANT'), find_dir('wholeBodyPOST')
print('ANT folder :', ANT_DIR)
print('POST folder:', POST_DIR)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')
for d in [ANT_DIR, POST_DIR]:
    files = sorted(os.listdir(d))
    imgs = [f for f in files if f.lower().endswith(IMG_EXT)]
    txts = [f for f in files if f.lower().endswith('.txt')]
    print(f'\n{os.path.basename(d)}: {len(imgs)} images, txt files: {txts}')
    print('  example images:', imgs[:3])
    for t in txts[:1]:
        with open(os.path.join(d, t)) as fh:
            print('  first lines of', t, ':')
            for _ in range(5):
                print('   ', repr(fh.readline()))

## Read labels and pair ANT and POST images per patient
ANT and POST images of a patient share the file name. A patient is labelled metastatic if either view is labelled metastatic.

In [ ]:
import numpy as np, pandas as pd

def read_folder(d):
    imgs = {os.path.splitext(f)[0]: os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(IMG_EXT)}
    labels = {}
    for t in glob.glob(os.path.join(d, '*.txt')):
        with open(t) as fh:
            for line in fh:
                toks = [x for x in re.split(r'[\s,;]+', line.strip()) if x]
                if len(toks) < 2:
                    continue
                name_tok = next((x for x in toks if x.lower().endswith(IMG_EXT)), toks[0])
                stem = os.path.splitext(os.path.basename(name_tok))[0]
                if stem in imgs and toks[-1] in ('0', '1'):
                    labels[stem] = int(toks[-1])
    return imgs, labels

ant_imgs, ant_lab = read_folder(ANT_DIR)
post_imgs, post_lab = read_folder(POST_DIR)
print(f'ANT : {len(ant_imgs)} images, {len(ant_lab)} with label')
print(f'POST: {len(post_imgs)} images, {len(post_lab)} with label')

rows = []
for pid in sorted(set(ant_lab) & set(post_lab)):
    rows.append(dict(patient_id=pid, ANT=ant_imgs[pid], POST=post_imgs[pid],
                     y=max(ant_lab[pid], post_lab[pid]), agree=ant_lab[pid] == post_lab[pid]))
bs = pd.DataFrame(rows)
if len(bs) == 0:
    raise SystemExit('No patient pairs were formed - check cells 3 and 4.')
print('\nPaired patients:', len(bs))
print('ANT and POST label identical:', f"{bs.agree.mean():.1%}")
print(bs.y.map({0: 'Normal', 1: 'Metastasis'}).value_counts())

if QUICK_TEST:
    bs = bs.groupby('y', group_keys=False).apply(lambda g: g.sample(min(len(g), 150), random_state=SEED)).reset_index(drop=True)
    print('\nQuick test: only', len(bs), 'patients')

## Pre-processing (same as for our cohort)
Grayscale; images with a dark background are inverted so that all images show dark bone on a white background; letterboxed into 512 x 176 without stretching. The prepared images are cached on Drive.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def prep(path):
    a = np.array(Image.open(path).convert('L'))
    border = np.concatenate([a[:5].ravel(), a[-5:].ravel(), a[:, :5].ravel(), a[:, -5:].ravel()])
    if np.median(border) < 128:
        a = 255 - a
    im = Image.fromarray(a)
    im.thumbnail((W, H), Image.BILINEAR)
    canvas = Image.new('L', (W, H), 255)
    canvas.paste(im, ((W - im.width) // 2, (H - im.height) // 2))
    return np.array(canvas)

cache = OUT + ('/bs80k_quick.npz' if QUICK_TEST else '/bs80k_prepared.npz')
if os.path.exists(cache):
    z = np.load(cache, allow_pickle=True)
    keep = dict(zip(z['ids'], range(len(z['ids']))))
    bs = bs[bs.patient_id.isin(keep)].reset_index(drop=True)
    ANT = z['ant'][[keep[p] for p in bs.patient_id]]
    POST = z['post'][[keep[p] for p in bs.patient_id]]
    print('Loaded from cache:', len(bs))
else:
    ANT = np.stack([prep(p) for p in bs.ANT])
    POST = np.stack([prep(p) for p in bs.POST])
    np.savez_compressed(cache, ids=bs.patient_id.values, ant=ANT, post=POST)
    print('Images prepared and saved:', len(bs))

fig, ax = plt.subplots(1, 8, figsize=(12, 6))
for j, lab in enumerate([0, 0, 1, 1]):
    i = bs.index[bs.y == lab][j % 2]
    ax[2*j].imshow(ANT[i], cmap='gray'); ax[2*j].set_title(('Normal' if lab == 0 else 'Metastasis') + '\nANT', fontsize=9)
    ax[2*j+1].imshow(POST[i], cmap='gray'); ax[2*j+1].set_title(('Normal' if lab == 0 else 'Metastasis') + '\nPOST', fontsize=9)
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(OUT + '/samples.png', dpi=150); plt.show()

## Patient-wise 80 / 10 / 10 split

In [ ]:
from sklearn.model_selection import train_test_split
bs['idx'] = np.arange(len(bs))
tr, tmp = train_test_split(bs, test_size=0.2, stratify=bs.y, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.5, stratify=tmp.y, random_state=SEED)
bs['split'] = 'train'
bs.loc[va.index, 'split'] = 'val'
bs.loc[te.index, 'split'] = 'test'
assert bs.groupby('patient_id').split.nunique().max() == 1
print('Leakage check: PASS\n')
print(pd.crosstab(bs.y.map({0: 'Normal', 1: 'Metastasis'}), bs.split, margins=True))
bs.drop(columns=['ANT', 'POST']).to_csv(OUT + ('/split_quick.csv' if QUICK_TEST else '/split.csv'), index=False)

## BS-80K duplicate audit (report only; training is not changed)
Checks whether a patient's ANT and POST are the same image and whether the same image appears under two patients, in particular across the training and test splits.

In [ ]:
def tiny_arr(a):
    a = a.astype(np.float32); m = a < 245
    if m.any():
        ys, xs = np.where(m); a = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    v = np.array(Image.fromarray(a.astype(np.uint8)).resize((32, 96), Image.BILINEAR), dtype=np.float32).ravel()
    return (v - v.mean()) / (v.std() + 1e-6)

N = len(bs)
X = np.stack([tiny_arr(a) for a in ANT] + [tiny_arr(a) for a in POST])     # 0..N-1 = ANT, N..2N-1 = POST
pid = np.concatenate([bs.patient_id.values, bs.patient_id.values]); spl = np.concatenate([bs.split.values, bs.split.values])
rows = []
for s0 in range(0, 2 * N, 1000):
    Cb = X[s0:s0 + 1000] @ X.T / X.shape[1]
    ii, jj = np.where(Cb > 0.99)
    for i, j in zip(ii + s0, jj):
        if j <= i:
            continue
        kind = 'same patient ANT=POST' if pid[i] == pid[j] else 'same image, different patients'
        if pid[i] == pid[j] and (i < N) == (j < N):
            continue
        rows.append(dict(kind=kind, patient_a=pid[i], split_a=spl[i], patient_b=pid[j], split_b=spl[j]))
aud = pd.DataFrame(rows, columns=['kind', 'patient_a', 'split_a', 'patient_b', 'split_b'])
aud.to_csv(OUT + '/bs80k_audit.csv', index=False)
print('BS-80K audit:')
print('  ANT = POST (same patient):', int((aud.kind == 'same patient ANT=POST').sum()))
cross = aud[aud.kind != 'same patient ANT=POST']
print('  Same image in different patients:', len(cross))
leak = cross[cross.split_a != cross.split_b]
print('  ... of which across different splits (LEAKAGE):', len(leak))
if len(leak):
    print(leak.groupby(['split_a', 'split_b']).size())
print('Saved:', OUT + '/bs80k_audit.csv')

## Dataset, augmentation and model

In [ ]:
import random, copy, time
import torch.nn as nn, torch.nn.functional as F
import timm
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix, recall_score)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.93, 1.07), fill=255),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.Grayscale(num_output_channels=3), T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Grayscale(num_output_channels=3), T.ToTensor(), T.Normalize(MEAN, STD)])

class ArrayDataset(Dataset):
    """ant/post: arrays of shape (N, H, W) uint8, y: labels"""
    def __init__(self, ant, post, y, tf):
        self.ant, self.post, self.y, self.tf = ant, post, np.asarray(y), tf
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return (self.tf(Image.fromarray(self.ant[i])), self.tf(Image.fromarray(self.post[i])), int(self.y[i]))

class DualViewNet(nn.Module):
    def __init__(self, name, n_classes, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(name, pretrained=pretrained, num_classes=0)
        self.backbone.eval()
        with torch.no_grad():
            f = self.backbone(torch.zeros(1, 3, H, W)).shape[1]
        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(2 * f, 256), nn.ReLU(),
                                  nn.Dropout(0.3), nn.Linear(256, n_classes))
    def forward(self, ant, post):
        return self.head(torch.cat([self.backbone(ant), self.backbone(post)], dim=1))

def loader(part, tf, shuffle=False):
    d = bs[bs.split == part]
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(ArrayDataset(ANT[d.idx.values], POST[d.idx.values], d.y.values, tf), BATCH,
                      shuffle=shuffle, drop_last=shuffle, num_workers=2, generator=g)

print('Ready')

## Training and evaluation functions

In [ ]:
def predict(model, dl):
    model.eval(); probs, ys = [], []
    with torch.no_grad(), torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
        for a, p, y in dl:
            out = model(a.to(DEVICE), p.to(DEVICE))
            probs.append(F.softmax(out.float(), 1).cpu()); ys.append(y)
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()

def metrics(probs, y):
    pred = probs.argmax(1)
    m = dict(accuracy=accuracy_score(y, pred), balanced_acc=balanced_accuracy_score(y, pred),
             macro_f1=f1_score(y, pred, average='macro'),
             sensitivity=recall_score(y, pred, pos_label=1, zero_division=0),
             specificity=recall_score(y, pred, pos_label=0, zero_division=0))
    try:
        m['auc'] = roc_auc_score(y, probs[:, 1])
    except ValueError:
        m['auc'] = float('nan')
    return m

def save_cm(y, probs, title, path):
    cm = confusion_matrix(y, probs.argmax(1), labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4)); ax.imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Normal', 'Metastasis'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['Normal', 'Metastasis'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(title)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()

def run(model_name):
    set_seed(SEED)
    tr_dl, va_dl, te_dl = loader('train', train_tf, True), loader('val', eval_tf), loader('test', eval_tf)
    ytr = bs[bs.split == 'train'].y.values
    counts = np.bincount(ytr, minlength=2)
    w = torch.tensor(len(ytr) / (2 * counts), dtype=torch.float32).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    model = DualViewNet(model_name, 2).to(DEVICE)
    opt = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': 1e-4},
                             {'params': model.head.parameters(), 'lr': 1e-3}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler(enabled=(DEVICE == 'cuda'))
    best_f1, best_state, wait, t0 = -1, None, 0, time.time()
    for ep in range(1, EPOCHS + 1):
        model.train(); total = 0
        for a, p, y in tr_dl:
            a, p, y = a.to(DEVICE), p.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
                loss = loss_fn(model(a, p), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            total += loss.item()
        sched.step()
        vf1 = metrics(*predict(model, va_dl))['macro_f1']
        print(f'  epoch {ep:2d} | train loss {total/len(tr_dl):.3f} | val macro-F1 {vf1:.3f} | {(time.time()-t0)/60:.1f} min')
        if vf1 > best_f1:
            best_f1, best_state, wait = vf1, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print('  early stopping'); break
    model.load_state_dict(best_state)
    probs, y = predict(model, te_dl)
    m = metrics(probs, y)
    m.update(model=model_name, best_val_f1=best_f1, epochs_run=ep, minutes=round((time.time() - t0) / 60, 1))
    save_cm(y, probs, f'BS-80K\n{model_name}', f'{OUT}/cm_bs80k_{model_name}.png')
    torch.save(best_state, f'{OUT}/model_bs80k_{model_name}.pt')
    return m

print('Functions ready')

## Training

In [ ]:
res_file = OUT + ('/quick_test.csv' if QUICK_TEST else '/results_bs80k.csv')
done = pd.read_csv(res_file) if os.path.exists(res_file) else pd.DataFrame(columns=['model'])
for model_name in MODELS:
    if (done['model'] == model_name).any():
        print('SKIP (already done):', model_name); continue
    print(f'\n===== BS-80K | {model_name} =====')
    m = run(model_name)
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in m.items()})
    done = pd.concat([done, pd.DataFrame([m])], ignore_index=True)
    done.to_csv(res_file, index=False)
    torch.cuda.empty_cache()
print('\nSab models mukammal!')

## Results

In [ ]:
r = pd.read_csv(res_file)
cols = ['model', 'accuracy', 'balanced_acc', 'macro_f1', 'auc', 'sensitivity', 'specificity']
print('BS-80K test results:')
display(r[cols].round(3))

own_file = BASE + '/prepared/results/results_v2.csv'
if not os.path.exists(own_file):
    own_file = BASE + '/prepared/results/results.csv'
if os.path.exists(own_file):
    own = pd.read_csv(own_file)
    own = own[own.task == 'sec_vs_normal'][['model', 'macro_f1', 'auc']]
    comp = own.merge(r[['model', 'macro_f1', 'auc']], on='model', suffixes=('_our_data', '_bs80k'))
    print('\nComparison (macro-F1 and AUC):')
    display(comp.round(3))
    comp.to_csv(OUT + '/comparison_our_vs_bs80k_v2.csv', index=False)
    ax = comp.set_index('model')[['macro_f1_our_data', 'macro_f1_bs80k']].plot(kind='bar', figsize=(9, 4), edgecolor='black')
    ax.set_ylabel('Test Macro-F1'); ax.set_ylim(0, 1.05); plt.xticks(rotation=15)
    ax.legend(['Our data (Secondary vs Normal)', 'BS-80K (Metastasis vs Normal)'])
    plt.title('Same models on two datasets'); plt.tight_layout()
    plt.savefig(OUT + '/comparison_our_vs_bs80k_v2.png', dpi=150); plt.show()

## Optional: cross-dataset test without retraining (exploratory; not used in the revised manuscript)

In [ ]:
split_own = pd.read_csv(BASE + '/prepared/split.csv')
own_te = split_own[(split_own.split == 'test') & split_own.label.isin(['SECONDARY', 'NORMAL'])]
own_ant = np.stack([np.array(Image.open(p).convert('L')) for p in own_te.ANT])
own_post = np.stack([np.array(Image.open(p).convert('L')) for p in own_te.POST])
own_y = (own_te.label == 'SECONDARY').astype(int).values
own_dl = DataLoader(ArrayDataset(own_ant, own_post, own_y, eval_tf), BATCH, num_workers=2)
bs_te_dl = loader('test', eval_tf)

cross = []
for name in MODELS:
    for src, wfile, dl, tgt in [('Our data', f'{BASE}/prepared/results/model_sec_vs_normal_{name}.pt', bs_te_dl, 'BS-80K'),
                                ('BS-80K', f'{OUT}/model_bs80k_{name}.pt', own_dl, 'Our data')]:
        if not os.path.exists(wfile):
            continue
        model = DualViewNet(name, 2, pretrained=False).to(DEVICE)
        model.load_state_dict(torch.load(wfile, map_location=DEVICE))
        m = metrics(*predict(model, dl))
        cross.append(dict(model=name, trained_on=src, tested_on=tgt, **{k: round(v, 3) for k, v in m.items()}))
        del model; torch.cuda.empty_cache()
cross = pd.DataFrame(cross)
display(cross)
cross.to_csv(OUT + '/cross_dataset_v2.csv', index=False)
print('All BS-80K results saved in:', OUT)